In [ ]:
-- LAB | Temporary Tables, Views and CTEs - Sakila Database
-- Customer Summary Report Creation

-- =====================================================
-- STEP 1: CREATE A VIEW
-- Create a view that summarizes rental information for each customer
-- =====================================================

-- Drop view if exists (for rerunning the script)
DROP VIEW IF EXISTS rental_summary;

-- Create the rental summary view
CREATE VIEW rental_summary AS
SELECT 
    c.customer_id,
    CONCAT(c.first_name, ' ', c.last_name) AS customer_name,
    c.email,
    COUNT(r.rental_id) AS rental_count
FROM customer c
LEFT JOIN rental r ON c.customer_id = r.customer_id
GROUP BY c.customer_id, c.first_name, c.last_name, c.email;

-- Verify the view creation
SELECT * FROM rental_summary 
ORDER BY rental_count DESC 
LIMIT 10;


-- =====================================================
-- STEP 2: CREATE A TEMPORARY TABLE
-- Create a Temporary Table that calculates the total amount paid by each customer
-- =====================================================

-- Drop temporary table if exists (for rerunning the script)
DROP TEMPORARY TABLE IF EXISTS customer_payment_summary;

-- Create the temporary table using the rental summary view
CREATE TEMPORARY TABLE customer_payment_summary AS
SELECT 
    rs.customer_id,
    rs.customer_name,
    rs.email,
    rs.rental_count,
    COALESCE(SUM(p.amount), 0) AS total_paid
FROM rental_summary rs
LEFT JOIN payment p ON rs.customer_id = p.customer_id
GROUP BY rs.customer_id, rs.customer_name, rs.email, rs.rental_count;

-- Verify the temporary table creation
SELECT * FROM customer_payment_summary 
ORDER BY total_paid DESC 
LIMIT 10;


-- =====================================================
-- STEP 3: CREATE A CTE AND CUSTOMER SUMMARY REPORT
-- Create a CTE that joins the rental summary View with the customer payment summary Temporary Table
-- Generate the final customer summary report
-- =====================================================

-- Customer Summary Report using CTE
WITH customer_summary_cte AS (
    SELECT 
        customer_name,
        email,
        rental_count,
        total_paid,
        CASE 
            WHEN rental_count > 0 THEN ROUND(total_paid / rental_count, 2)
            ELSE 0
        END AS average_payment_per_rental
    FROM customer_payment_summary
)
SELECT 
    customer_name,
    email,
    rental_count,
    total_paid,
    average_payment_per_rental
FROM customer_summary_cte
ORDER BY total_paid DESC;


-- =====================================================
-- ALTERNATIVE APPROACH: All-in-One CTE Solution
-- This demonstrates how the entire report could be created using only CTEs
-- =====================================================

WITH rental_info AS (
    -- Get rental count per customer
    SELECT 
        c.customer_id,
        CONCAT(c.first_name, ' ', c.last_name) AS customer_name,
        c.email,
        COUNT(r.rental_id) AS rental_count
    FROM customer c
    LEFT JOIN rental r ON c.customer_id = r.customer_id
    GROUP BY c.customer_id, c.first_name, c.last_name, c.email
),
payment_info AS (
    -- Get total payments per customer
    SELECT 
        customer_id,
        COALESCE(SUM(amount), 0) AS total_paid
    FROM payment
    GROUP BY customer_id
),
customer_summary AS (
    -- Combine rental and payment information
    SELECT 
        ri.customer_name,
        ri.email,
        ri.rental_count,
        COALESCE(pi.total_paid, 0) AS total_paid,
        CASE 
            WHEN ri.rental_count > 0 THEN ROUND(COALESCE(pi.total_paid, 0) / ri.rental_count, 2)
            ELSE 0
        END AS average_payment_per_rental
    FROM rental_info ri
    LEFT JOIN payment_info pi ON ri.customer_id = pi.customer_id
)
SELECT * FROM customer_summary
ORDER BY total_paid DESC;


-- =====================================================
-- VERIFICATION QUERIES
-- Additional queries to verify our results and explore the data
-- =====================================================

-- 1. Check total number of customers
SELECT COUNT(*) AS total_customers FROM customer;

-- 2. Verify rental counts match between view and direct query
SELECT 
    'View Results' AS source,
    SUM(rental_count) AS total_rentals,
    COUNT(*) AS customers_with_rentals
FROM rental_summary
WHERE rental_count > 0

UNION ALL

SELECT 
    'Direct Query' AS source,
    COUNT(*) AS total_rentals,
    COUNT(DISTINCT customer_id) AS customers_with_rentals
FROM rental;

-- 3. Verify payment totals
SELECT 
    'Temp Table Results' AS source,
    SUM(total_paid) AS total_payments,
    COUNT(*) AS customers_with_payments
FROM customer_payment_summary
WHERE total_paid > 0

UNION ALL

SELECT 
    'Direct Query' AS source,
    SUM(amount) AS total_payments,
    COUNT(DISTINCT customer_id) AS customers_with_payments
FROM payment;

-- 4. Top 10 customers by rental count
SELECT 
    customer_name,
    rental_count,
    total_paid,
    average_payment_per_rental
FROM customer_payment_summary
ORDER BY rental_count DESC
LIMIT 10;

-- 5. Top 10 customers by total paid
SELECT 
    customer_name,
    rental_count,
    total_paid,
    CASE 
        WHEN rental_count > 0 THEN ROUND(total_paid / rental_count, 2)
        ELSE 0
    END AS average_payment_per_rental
FROM customer_payment_summary
ORDER BY total_paid DESC
LIMIT 10;

-- 6. Customers with highest average payment per rental
SELECT 
    customer_name,
    rental_count,
    total_paid,
    CASE 
        WHEN rental_count > 0 THEN ROUND(total_paid / rental_count, 2)
        ELSE 0
    END AS average_payment_per_rental
FROM customer_payment_summary
WHERE rental_count > 0
ORDER BY (total_paid / rental_count) DESC
LIMIT 10;

-- 7. Summary statistics
SELECT 
    COUNT(*) AS total_customers,
    SUM(rental_count) AS total_rentals,
    SUM(total_paid) AS total_revenue,
    AVG(rental_count) AS avg_rentals_per_customer,
    AVG(total_paid) AS avg_payment_per_customer,
    MAX(rental_count) AS max_rentals_by_customer,
    MAX(total_paid) AS max_payment_by_customer
FROM customer_payment_summary;


-- =====================================================
-- CLEANUP (Optional)
-- Uncomment these lines if you want to clean up the objects created
-- =====================================================

-- DROP VIEW IF EXISTS rental_summary;
-- DROP TEMPORARY TABLE IF EXISTS customer_payment_summary;


-- =====================================================
-- NOTES AND EXPLANATIONS
-- =====================================================

/*
VIEWS:
- rental_summary view provides a reusable, virtual table
- Automatically updates when underlying data changes
- Good for frequently used complex queries
- Can be used by multiple users/applications

TEMPORARY TABLES:
- customer_payment_summary stores intermediate results physically
- Exists only for the current session
- Can be indexed for better performance on large datasets
- Useful for complex multi-step calculations

CTEs (Common Table Expressions):
- Provide readable, modular query structure
- Exist only for the duration of a single query
- Can be recursive (though not used in this example)
- Great for breaking down complex logic into manageable parts

PERFORMANCE CONSIDERATIONS:
- Views: No storage overhead, but recalculated each time
- Temporary Tables: Storage overhead, but pre-calculated results
- CTEs: No storage, calculated once per query execution
*/